# Supplemental report
Attach the release code dataset, supplemental and intermediate-dose input bundles, and the six saved result ZIPs listed in the artifact index. Run on CPU.


In [ ]:
import time
SESSION_STARTED=time.monotonic()
import importlib.metadata as metadata
import subprocess, sys
required = {'numpy':'2.1.3','pandas':'2.3.3','scipy':'1.16.3','scikit-learn':'1.6.1','timm':'1.0.29','Pillow':'13.0.0','huggingface_hub':'1.33.0','safetensors':'0.8.0','matplotlib':'3.10.8'}
changes=[]
for name,version in required.items():
    try: installed=metadata.version(name)
    except metadata.PackageNotFoundError: installed=None
    if installed!=version: changes.append(f'{name}=={version}')
if changes: subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps',*changes],check=True)
print('Pinned analysis packages ready; Kaggle torch and torchvision retained.')


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import hashlib,json,shutil,stat,sys
EXPECTED_SHA = "8d6fe11c06d5ce9af25f227d99fd63fdc4854dba924e01a584b0126efad4c5f5"
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
search=Path('/kaggle/input');markers=list(search.rglob('supplemental_manifest.json'))
if not any(file_sha(p)==EXPECTED_SHA for p in markers):
    for archive in search.rglob('MSLD_Supplemental_Inputs.zip'):
        destination=Path('/tmp/msld_supplemental_bootstrap')/file_sha(archive)
        with ZipFile(archive) as z:
            if sum(i.file_size for i in z.infolist())>100*2**20:raise ValueError('Expected the small supplemental input ZIP.')
            for info in z.infolist():
                rel=PurePosixPath(info.filename.replace('\\','/'));target=destination.joinpath(*rel.parts).resolve()
                if rel.is_absolute() or '..' in rel.parts or any(':' in a for a in rel.parts) or stat.S_ISLNK(info.external_attr>>16) or not target.is_relative_to(destination.resolve()):raise ValueError('Unsafe ZIP path')
                if info.is_dir():target.mkdir(parents=True,exist_ok=True)
                else:
                    target.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(info) as src,target.open('wb') as dst:shutil.copyfileobj(src,dst)
        markers+=list(destination.rglob('supplemental_manifest.json'))
matches=[p for p in markers if file_sha(p)==EXPECTED_SHA]
if not matches:raise FileNotFoundError('Attach MSLD_Supplemental_Inputs.zip as a private Kaggle dataset.')
INPUTS=sorted(matches)[0].parent
index=json.loads((INPUTS/'supplemental_manifest.json').read_text())
for rel,rec in index['files'].items():
    p=(INPUTS/rel).resolve()
    if not p.is_relative_to(INPUTS.resolve()) or not p.is_file() or p.stat().st_size!=rec['bytes'] or file_sha(p)!=rec['sha256']:raise ValueError('Missing or changed input: '+rel)
CODE=Path('/kaggle/working/supplemental_code');CODE.mkdir(exist_ok=True)
for source in (INPUTS/'code').glob('*.py'):shutil.copyfile(source,CODE/source.name)
print('Supplemental package verified. Analysis plan:',file_sha(INPUTS/'analysis_plan.json'))

EXTENSION_SHA='c464d0e82def618ee5caadf394c6894b52184d5f5e6b6a27994a6fed1157d371'
markers=list(search.rglob('extension_manifest.json'))
if not any(file_sha(p)==EXTENSION_SHA for p in markers):
    for archive in search.rglob('MSLD_Additional_Checks_Inputs.zip'):
        target=Path('/tmp/msld_additional_inputs')/file_sha(archive)
        with ZipFile(archive) as z:
            if sum(i.file_size for i in z.infolist())>100*1024*1024:raise ValueError('Additional input archive too large.')
            for info in z.infolist():
                rel=PurePosixPath(info.filename.replace('\\','/'))
                if rel.is_absolute() or '..' in rel.parts or any(':' in p for p in rel.parts) or stat.S_ISLNK(info.external_attr>>16):raise ValueError('Unsafe additional archive.')
                dest=target.joinpath(*rel.parts)
                if info.is_dir():dest.mkdir(parents=True,exist_ok=True)
                else:
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(info) as src,dest.open('wb') as dst:shutil.copyfileobj(src,dst)
        markers.extend(target.rglob('extension_manifest.json'))
match=[p for p in markers if file_sha(p)==EXTENSION_SHA]
if not match:raise FileNotFoundError('Attach NEW MSLD_Additional_Checks_Inputs.zip as private msld-additional-checks-inputs.')
EXTRA=sorted(match)[0].parent
extra_index=json.loads((EXTRA/'extension_manifest.json').read_text())
for rel,rec in extra_index['files'].items():
    p=(EXTRA/rel).resolve()
    if not p.is_relative_to(EXTRA.resolve()) or not p.is_file() or p.stat().st_size!=rec['bytes'] or file_sha(p)!=rec['sha256']:raise ValueError('Additional input missing/changed: '+rel)
EXTRA_CODE=Path('/kaggle/working/additional_code');EXTRA_CODE.mkdir(exist_ok=True)
for p in (EXTRA/'code').glob('*.py'):shutil.copyfile(p,EXTRA_CODE/p.name)
print('Additional analysis plan and code verified.')


In [ ]:
import os
SCOPED_REPORT = Path('/kaggle/working/release_overlays/final_report_six_zip.py')
from pathlib import Path
import subprocess
import sys

code_tools = sorted(Path('/kaggle/input').rglob('materialize_workflow.py'))
code_tools = [p for p in code_tools if (p.parents[1] / 'reproducibility/code_sources.csv').is_file()]
if len(code_tools) != 1:
    raise FileNotFoundError('Attach one extracted release code dataset. See docs/reproduce.md.')
subprocess.run([sys.executable, str(code_tools[0]), 'supplemental_report', str(SCOPED_REPORT.parent)], check=True)
env={**os.environ,'PYTHONPATH':str(EXTRA_CODE)+os.pathsep+str(CODE)+os.pathsep+os.environ.get('PYTHONPATH','')}
base=[sys.executable,str(SCOPED_REPORT),'--inputs',str(INPUTS),'--extension',str(EXTRA)]
print('Six-ZIP reporting v2 installed. Both original queue status schemas are supported.')


In [ ]:
subprocess.run([*base,'--check-only'],env=env,check=True)


In [ ]:
subprocess.run(base,env=env,check=True)
